# D1a: GMM Clustering

In [ ]:
# ── Setup: clone repo + install deps ─────────────────────────
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
!pip install hdbscan umap-learn -q

import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')


In [ ]:
# ── Imports ──────────────────────────────────────────────────
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
    normalized_mutual_info_score, f1_score, classification_report, confusion_matrix)
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')


In [ ]:
# ── Cycle extraction (matching Helene's tsne_v2.py) ────────
# Key: double savgol, 300 deg/s min peak height, resample boundaries,
# overlap-based pairing, omega in rad/s in cycle matrix

CONFIG = {
    'FS': 50.0,
    'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21,
    'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0,   # strict: reject weak peaks
    'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0,
    'TARGET_LEN': 64,
    'MIN_CYCLE_SAMPLES': 10,
}

KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}
EXCLUDE_LABELS = {None, 'CW', 'CCW'}  # never use CW/CCW as classes

_EXACT = {'underhand':'underhand','overhand':'overhand','dragon_roll':'dragon_roll',
          'sneak_underhand':'sneak_underhand','sneak_overhand':'sneak_overhand','idle':None}
_PRULES = [
    (re.compile(r'^us', re.I), 'sneak_underhand'), (re.compile(r'^os', re.I), 'sneak_overhand'),
    (re.compile(r'^u', re.I), 'underhand'), (re.compile(r'^o', re.I), 'overhand'),
    (re.compile(r'^fb', re.I), 'dragon_roll'), (re.compile(r'^bf', re.I), 'dragon_roll'),
    (re.compile(r'^cw$', re.I), 'CW'), (re.compile(r'^ccw$', re.I), 'CCW'),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
def map_label(raw):
    raw = raw.strip()
    if raw.lower() in _EXACT: return _EXACT[raw.lower()]
    for pat, c in _PRULES:
        if pat.match(raw): return c
    return None

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w','ay_w','az_w']].values
    omega = df[['gx','gy','gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag_deg = np.linalg.norm(omega, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7: return [], mag_deg, np.array([], dtype=int)
    win = int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if win % 2 == 0: win += 1
    win = max(5, min(win, n if n % 2 == 1 else n - 1))
    poly = max(1, min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']), win - 2))
    mag_s = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    mag_s = savgol_filter(mag_s, window_length=win, polyorder=poly, mode='interp')
    peaks, _ = find_peaks(mag_s, distance=max(1, int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)),
                          prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],
                          height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(peaks) == 0: return [], mag_s, peaks
    cycles = []
    for i, p in enumerate(peaks):
        left = 0 if i == 0 else (peaks[i-1] + p) // 2
        right = (len(t)-1) if i == len(peaks)-1 else (p + peaks[i+1]) // 2
        if right > left and (right - left) >= CONFIG['MIN_CYCLE_SAMPLES']:
            cycles.append((left, right))
    return cycles, mag_s, peaks

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T.astype(np.float32)  # (12, 64)

print('Functions defined')


In [ ]:
# ── Load all cycles + labels ─────────────────────────────────

all_matrices = []; all_labels = []; all_groups = []; session_names = []

def load_session_cycles(d0_path, d1_path, name, label='unlabeled', windows=None):
    df0, df1 = pd.read_csv(d0_path), pd.read_csv(d1_path)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    cyc0, _, _ = detect_cycles(t0, om0, CONFIG['FS'])
    cyc1, _, _ = detect_cycles(t1, om1, CONFIG['FS'])
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if windows is not None:
        fp0, fp1 = [], []
        for (s0,e0),(s1,e1) in zip(p0,p1):
            t_mid = (t0[s0]+t0[e0])/2
            if any(ws <= t_mid < we for ws, we in windows):
                fp0.append((s0,e0)); fp1.append((s1,e1))
        p0, p1 = fp0, fp1
    grp = name.split('/')[0] if '/' in name else name
    count = 0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm); all_labels.append(label); all_groups.append(grp)
        count += 1
    if count > 0: session_names.append(name)
    return count

# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv'))):
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    label = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            label = 'underhand' if pat == 'underhand_default' else pat; break
    n = load_session_cycles(d0, d1, stem, label)
    if n > 0: print(f'  {stem:<50s} {label:<20s} {n:>4d}')

# Heterogeneous
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f: data = _json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            canon = map_label(seg.get('label', ''))
            if canon in EXCLUDE_LABELS: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[canon].append((float(s), float(e)))
        for label, windows in sorted(wbl.items()):
            n = load_session_cycles(d0, d1, sname + '/' + label, label, windows)
            if n > 0: print(f'  {sname}/{label:<20s} {"":>29s} {n:>4d}')

X_raw = np.array([cm.ravel() for cm in all_matrices])  # (N, 768)
y_all = np.array(all_labels)
g_all = np.array(all_groups)

# Masks
labeled_mask = np.array([l not in ('unlabeled', 'CW', 'CCW') for l in y_all])
X_labeled = X_raw[labeled_mask]
y_labeled = y_all[labeled_mask]
g_labeled = g_all[labeled_mask]

print('\nTotal: ' + str(len(X_raw)) + ' cycles (' + str(X_raw.shape[1]) + 'D)')
print('Labeled: ' + str(labeled_mask.sum()) + ', Unlabeled: ' + str((~labeled_mask).sum()))
print('Label distribution:')
for lab, cnt in sorted(Counter(y_labeled).items(), key=lambda x: -x[1]):
    print(f'  {lab:<20s}: {cnt:>5d}')
print('Groups: ' + str(len(np.unique(g_labeled))))


In [ ]:
# ── PCA + StandardScaler ──────────────────────────────────────

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_labeled_scaled = X_scaled[labeled_mask]

pca_full = PCA(n_components=min(50, X_scaled.shape[1], X_scaled.shape[0]-1), svd_solver='full')
X_pca = pca_full.fit_transform(X_scaled)
X_labeled_pca = X_pca[labeled_mask]
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
print('PCA: 768D -> ' + str(X_pca.shape[1]) + 'D (' + str(round(cumvar[-1]*100,1)) + '% var)')


## D1a: Gaussian Mixture Model Clustering

In [ ]:
from sklearn.mixture import GaussianMixture

K_VALUES = [5, 8, 10, 11, 12, 13, 15]
gmm_results = {}

print('GMM clustering on ' + str(X_pca.shape[1]) + 'D PCA:')
for k in K_VALUES:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, n_init=5)
    cl = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, cl)
    bic = gmm.bic(X_pca)
    aic = gmm.aic(X_pca)
    gmm_results[k] = {'labels': cl, 'sil': sil, 'bic': bic, 'aic': aic}
    ari = adjusted_rand_score(y_all[labeled_mask], cl[labeled_mask])
    nmi = normalized_mutual_info_score(y_all[labeled_mask], cl[labeled_mask])
    print(f'  k={k:>2d}: sil={sil:.3f}, ARI={ari:.3f}, NMI={nmi:.3f}, BIC={bic:.0f}')


In [ ]:
# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sils = [gmm_results[k]['sil'] for k in K_VALUES]
bics = [gmm_results[k]['bic'] for k in K_VALUES]
aics = [gmm_results[k]['aic'] for k in K_VALUES]
axes[0].plot(K_VALUES, sils, 'o-'); axes[0].set_title('Silhouette'); axes[0].set_xticks(K_VALUES)
axes[1].plot(K_VALUES, bics, 's-', color='red'); axes[1].set_title('BIC (lower=better)'); axes[1].set_xticks(K_VALUES)
axes[2].plot(K_VALUES, aics, '^-', color='green'); axes[2].set_title('AIC (lower=better)'); axes[2].set_xticks(K_VALUES)
plt.tight_layout(); plt.show()


In [ ]:
# t-SNE visualization for best k by BIC
best_k_bic = K_VALUES[np.argmin(bics)]
best_k_sil = K_VALUES[np.argmax(sils)]
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.scatter(X_tsne[:, 0], X_tsne[:, 1], c=gmm_results[best_k_bic]['labels'], cmap='tab20', s=5, alpha=0.5)
ax1.set_title(f'GMM k={best_k_bic} (best BIC)')
ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=gmm_results[best_k_sil]['labels'], cmap='tab20', s=5, alpha=0.5)
ax2.set_title(f'GMM k={best_k_sil} (best silhouette)')
plt.tight_layout(); plt.show()


In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────
print('='*60)
print('D1a: GMM CLUSTERING SUMMARY')
print('='*60)
print(f'Total cycles: {len(X_raw)}, PCA dim: {X_pca.shape[1]}')
print(f'Labeled: {labeled_mask.sum()}, Unlabeled: {(~labeled_mask).sum()}')
print(f'Best k by BIC: {best_k_bic} (BIC={min(bics):.0f})')
print(f'Best k by silhouette: {best_k_sil} (sil={max(sils):.3f})')
for k in K_VALUES:
    r = gmm_results[k]
    ari = adjusted_rand_score(y_all[labeled_mask], r['labels'][labeled_mask])
    nmi = normalized_mutual_info_score(y_all[labeled_mask], r['labels'][labeled_mask])
    print(f'  k={k:>2d}: sil={r["sil"]:.3f} ARI={ari:.3f} NMI={nmi:.3f} BIC={r["bic"]:.0f} AIC={r["aic"]:.0f}')
# Purity for best k by BIC
cl = gmm_results[best_k_bic]['labels']
total_known = labeled_mask.sum()
correct = sum(max(Counter(y_all[(cl == c) & labeled_mask]).values(), default=0) for c in range(best_k_bic))
print(f'Purity at k={best_k_bic}: {correct/total_known:.3f}')
